# CXR-LT 2023

This notebook is the merged CXR-LT 2023 examination pass. It combines the original split/schema checks with the follow-on study, timeline, co-occurrence, view-conditioned, image-path, and report-linkage checks.

It reports both split-level summaries and global summaries that ignore the train/development/test split. Bar charts annotate each bar with both count and percentage: vertical bars show labels above the bar, and horizontal bars show labels at the bar end.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from utils import *

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

root_dir, data_dir = get_notebook_paths()
cxr_lt_2023_dir = data_dir / "CXR-LT" / "cxr-lt-multi-label-long-tailed-classification-on-chest-x-rays-2.0.0" / "cxr-lt-2023"
mimic_cxr_dir = data_dir / "MIMIC-CXR"
mimic_cxr_jpg_dir = data_dir / "MIMIC-CXR-JPG"

print(f"root_dir: {root_dir}")
print(f"cxr_lt_2023_dir: {cxr_lt_2023_dir}")
print(f"mimic_cxr_dir exists: {mimic_cxr_dir.exists()}")
print(f"mimic_cxr_jpg_dir exists: {mimic_cxr_jpg_dir.exists()}")


In [ ]:
CSV_FILES = {
    "train": "train.csv",
    "development": "development.csv",
    "test": "test.csv",
    "development_sample_submission": "development_sample_submission.csv",
    "test_sample_submission": "test_sample_submission.csv",
}

ANALYSIS_SPLIT_NAMES = ["train", "development", "test"]
ID_COLUMNS = BASE_ID_COLUMNS + ["path"]

datasets = load_csv_map(cxr_lt_2023_dir, CSV_FILES)

train_df = datasets["train"]
dev_df = datasets["development"]
test_df = datasets["test"]
dev_sample_sub_df = datasets["development_sample_submission"]
test_sample_sub_df = datasets["test_sample_submission"]

analysis_splits = {split_name: datasets[split_name] for split_name in ANALYSIS_SPLIT_NAMES}
submission_splits = {
    "development_sample_submission": dev_sample_sub_df,
    "test_sample_submission": test_sample_sub_df,
}

global_df = pd.concat(
    [df.assign(source_split=split_name) for split_name, df in analysis_splits.items()],
    ignore_index=True,
)
global_frames = {"global": global_df}

label_cols = label_columns(train_df)

print(f"Loaded {len(datasets)} files")
print(f"Detected {len(label_cols)} labels")
print(f"Global analysis rows: {len(global_df):,}")
label_cols


## Reusable helpers

Shared helpers now live in `utils.py` and are imported above.


## 1) Quick Split Overview

Start here to answer: how big are the splits, what fields exist, and where is metadata incomplete?


In [ ]:
for split_name, df in analysis_splits.items():
    preview_dataset(split_name, df)

split_overview_df = build_split_overview(analysis_splits, label_cols)
display(split_overview_df)


## 2) Global Dataset Statistics

These summaries disregard train/development/test and treat the CXR-LT 2023 labeled rows as one dataset.


In [ ]:
global_overview_df = build_split_overview(global_frames, label_cols).rename(columns={"split": "scope"})
global_label_summary_df = summarize_labels(global_df, "global", label_cols)
global_view_position_summary_df = summarize_view_positions(global_df, "global", top_n=12).rename(columns={"split": "scope"})
global_no_finding_conflicts_df = summarize_no_finding_conflicts(global_frames, label_cols).rename(columns={"split": "scope"})

display(global_overview_df)

print("Most common global labels")
display(global_label_summary_df.head(10))

print("Rarest global labels")
display(global_label_summary_df.tail(10))

print("Global view-position distribution")
display(global_view_position_summary_df)

print("Global No Finding conflicts")
display(global_no_finding_conflicts_df)

plot_global_label_prevalence(global_df, label_cols, top_n=26)
plot_global_view_positions(global_df, top_n=8)


## 3) Label Imbalance And Multi-Label Density

This is the part that tells you why the task is called long-tailed. Pay attention to the rarest labels and to how many findings co-occur on each image.


In [ ]:
label_summary_df = pd.concat(
    [summarize_labels(df, split_name, label_cols) for split_name, df in analysis_splits.items()],
    ignore_index=True,
)
label_summary_with_global_df = pd.concat([label_summary_df, global_label_summary_df], ignore_index=True)

display(label_summary_with_global_df.query("split == 'global'").head(10))
display(label_summary_with_global_df.query("split == 'global'").tail(10))

display(label_summary_df.query("split == 'train'").head(10))
display(label_summary_df.query("split == 'train'").tail(10))

plot_label_prevalence(analysis_splits, label_cols, reference_df=global_df, top_n=26)
plot_labels_per_image(analysis_splits, label_cols, include_global=True)

display(summarize_no_finding_conflicts(analysis_splits, label_cols))


## 4) View Metadata And Split Leakage Checks

For modeling, it helps to know whether view-position distribution shifts across splits and whether the same patient or study appears in multiple splits.


In [ ]:
view_position_summary_df = pd.concat(
    [summarize_view_positions(df, split_name) for split_name, df in analysis_splits.items()],
    ignore_index=True,
)
view_position_summary_with_global_df = pd.concat(
    [view_position_summary_df, global_view_position_summary_df.rename(columns={"scope": "split"})],
    ignore_index=True,
)

display(view_position_summary_with_global_df)
plot_view_positions(analysis_splits, reference_df=global_df, top_n=6)

for overlap_key in ["subject_id", "study_id", "dicom_id"]:
    print(f"\nOverlap for {overlap_key}")
    display(compute_overlap(analysis_splits, overlap_key))


## 5) Sample Submission Sanity Checks

These files are handy for confirming the label order expected by the benchmark and for checking score ranges before writing an inference pipeline.


In [ ]:
submission_summary_df = pd.concat(
    [summarize_submission_file(df, split_name, label_cols) for split_name, df in submission_splits.items()],
    ignore_index=True,
)
display(submission_summary_df)

for split_name, df in submission_splits.items():
    print(f"\n{split_name}")
    display(df.head(3))


## 6) Study-Level Aggregation

This checks whether a `study_id` usually has a frontal view, a lateral view, or both. This matters for multi-view fusion because the image-level rows are not the same unit as a clinical exam.


In [ ]:
study_table_df, study_view_summary_df = summarize_study_views(analysis_splits, label_cols)
global_study_table_df, global_study_view_summary_df = summarize_study_views(global_frames, label_cols)
global_study_view_summary_df = global_study_view_summary_df.rename(columns={"split": "scope"})

display(study_view_summary_df)
display(global_study_view_summary_df)

display(
    global_study_table_df
    [
        [
            "subject_id",
            "study_id",
            "image_count",
            "view_combo",
            "has_frontal_and_lateral",
            "positive_labels_per_study",
        ]
    ]
    .head(10)
)

plot_top_view_combos(study_table_df, top_n=12)
plot_global_top_view_combos(global_study_table_df, top_n=12)


## 7) Patient Timeline Behavior

The CXR-LT CSVs do not include acquisition time, so this joins MIMIC-CXR-JPG metadata when available. MIMIC dates are shifted, but within-patient ordering and intervals are still useful for repeat-exam analysis.


In [ ]:
metadata_df, metadata_path = load_first_existing(
    [
        mimic_cxr_jpg_dir / "mimic-cxr-2.0.0-metadata.csv.gz",
        mimic_cxr_jpg_dir / "mimic-cxr-2.0.0-metadata.csv",
    ]
)

if metadata_df is None:
    print("MIMIC-CXR-JPG metadata was not found. Timeline cells will fall back to study_id ordering.")
    metadata_columns = ["dicom_id", "study_datetime"]
    metadata_for_join = pd.DataFrame(columns=metadata_columns)
else:
    print(f"Loaded metadata: {metadata_path}")
    metadata_df["study_datetime"] = parse_mimic_study_datetime(metadata_df)
    metadata_columns = [
        column
        for column in [
            "dicom_id",
            "StudyDate",
            "StudyTime",
            "study_datetime",
            "PerformedProcedureStepDescription",
            "ProcedureCodeSequence_CodeMeaning",
            "Rows",
            "Columns",
        ]
        if column in metadata_df.columns
    ]
    metadata_for_join = metadata_df[metadata_columns].copy()

all_labeled_images_df = global_df.merge(metadata_for_join, on="dicom_id", how="left")

print(f"Rows with parsed study_datetime: {all_labeled_images_df['study_datetime'].notna().sum():,} / {len(all_labeled_images_df):,}")
display(all_labeled_images_df.head(3))


In [ ]:
study_timeline_df = build_study_timeline(all_labeled_images_df, label_cols, scope_columns=["source_split"])
global_study_timeline_df = build_study_timeline(all_labeled_images_df, label_cols)

patient_timeline_summary_df, split_patient_summary_df = summarize_patient_timelines(
    study_timeline_df,
    scope_columns=["source_split"],
)
global_patient_timeline_summary_df, global_patient_summary_df = summarize_patient_timelines(global_study_timeline_df)

split_patient_summary_df = split_patient_summary_df.rename(columns={"source_split": "split"})

display(split_patient_summary_df)
display(global_patient_summary_df)

display(
    global_patient_timeline_summary_df
    .sort_values(["study_count", "span_days"], ascending=[False, False])
    .head(10)
)


In [ ]:
top_patient_row = (
    global_patient_timeline_summary_df
    .sort_values(["study_count", "span_days"], ascending=[False, False])
    .head(1)
)

if top_patient_row.empty:
    print("No patient timeline rows available.")
else:
    top_subject_id = top_patient_row.iloc[0]["subject_id"]
    print(f"Example repeat-exam patient: subject_id={top_subject_id}")
    display(
        global_study_timeline_df
        .query("subject_id == @top_subject_id")
        [
            [
                "subject_id",
                "study_id",
                "study_datetime",
                "days_since_previous_study",
                "image_count",
                "view_combo",
                "positive_labels_per_study",
            ]
        ]
    )


## 8) Global Label Co-Occurrence And Correlations

These checks disregard split and use the complete labeled CXR-LT 2023 dataset. The correlation heatmap uses Pearson correlation on binary labels, which is the phi coefficient for two binary variables.


In [ ]:
global_corr_df = global_df[label_cols].corr()
pair_corr_df = label_pair_correlation_table(global_df, label_cols)

plt.figure(figsize=(14, 12))
sns.heatmap(global_corr_df, cmap="vlag", center=0, vmin=-1, vmax=1, square=True, linewidths=0.2)
plt.title("Global label correlation matrix")
plt.tight_layout()
plt.show()

print("Most positively correlated label pairs")
display(pair_corr_df.sort_values("correlation", ascending=False).head(15))

print("Most negatively correlated label pairs")
display(pair_corr_df.sort_values("correlation", ascending=True).head(15))


In [ ]:
global_cooccurrence_df = global_df[label_cols].T.dot(global_df[label_cols])
label_order = global_df[label_cols].sum().sort_values(ascending=False).index

plt.figure(figsize=(14, 12))
sns.heatmap(
    global_cooccurrence_df.loc[label_order, label_order],
    cmap="mako",
    square=True,
    linewidths=0.2,
)
plt.title("Global label co-occurrence counts")
plt.tight_layout()
plt.show()

display(global_cooccurrence_df.loc[label_order, label_order].iloc[:10, :10])


## 9) View-Conditioned Prevalence

This compares label rates for `AP`, `PA`, and `LATERAL`. It is useful for checking whether a label is view-dependent enough that frontal and lateral encoders should be treated separately.


In [ ]:
view_prevalence_df = view_conditioned_prevalence(analysis_splits, label_cols)
global_view_prevalence_df = view_conditioned_prevalence(global_frames, label_cols)

view_counts_df = (
    view_prevalence_df[["split", "ViewPosition", "row_count"]]
    .drop_duplicates()
    .sort_values(["split", "ViewPosition"])
)
global_view_counts_df = (
    global_view_prevalence_df[["split", "ViewPosition", "row_count"]]
    .drop_duplicates()
    .rename(columns={"split": "scope"})
    .sort_values(["scope", "ViewPosition"])
)

display(view_counts_df)
display(global_view_counts_df)

top_global_labels = global_df[label_cols].sum().sort_values(ascending=False).head(12).index.tolist()
plot_view_conditioned_prevalence(
    global_view_prevalence_df,
    label_order=top_global_labels,
    title="Global label prevalence by view position",
)

display(
    global_view_prevalence_df
    .query("label in @top_global_labels")
    .pivot(index="label", columns="ViewPosition", values="positive_rate_pct")
    .loc[top_global_labels]
    .round(3)
)


## 10) Image Availability And Path Sanity

CXR-LT `path` values are relative to the MIMIC-CXR-JPG root. This checks whether the expected JPG files are present locally before dataloader work.


In [ ]:
image_root = mimic_cxr_jpg_dir
print(f"image_root: {image_root}")
print(f"image_root exists: {image_root.exists()}")
print(f"image files directory exists: {(image_root / 'files').exists()}")

image_path_summary_df = summarize_image_paths(analysis_splits, image_root)
global_image_path_summary_df = summarize_image_paths(global_frames, image_root).rename(columns={"split": "scope"})

display(image_path_summary_df)
display(global_image_path_summary_df)

missing_global_paths = global_df.loc[
    ~global_df["path"].astype(str).map(lambda value: resolve_relative_path(image_root, value).is_file()),
    ["source_split", "dicom_id", "subject_id", "study_id", "ViewPosition", "path"],
].head(10)

print("Sample missing global image paths")
display(missing_global_paths)


## 11) Text Linkage Readiness

This joins CXR-LT studies to the MIMIC-CXR study list. A successful index join means the label row can be paired to a report path. File existence is a separate check because this workstation may have metadata without the report text tree.


In [ ]:
study_list_df, study_list_path = load_first_existing(
    [
        mimic_cxr_dir / "cxr-study-list.csv.gz",
        mimic_cxr_dir / "cxr-study-list.csv",
    ]
)

if study_list_df is None:
    print("MIMIC-CXR study list was not found. Report linkage cannot be checked.")
    report_link_summary_df = pd.DataFrame()
    global_report_link_summary_df = pd.DataFrame()
    linked_reports_df = pd.DataFrame()
    global_linked_reports_df = pd.DataFrame()
else:
    print(f"Loaded study list: {study_list_path}")
    (
        report_link_summary_df,
        global_report_link_summary_df,
        linked_reports_df,
        global_linked_reports_df,
    ) = build_report_linkage(analysis_splits, global_df, study_list_df, mimic_cxr_dir)

    print(f"report files directory exists: {(mimic_cxr_dir / 'files').exists()}")
    display(report_link_summary_df)
    display(global_report_link_summary_df)
    display(linked_reports_df.head(10))


In [ ]:
if global_linked_reports_df.empty or not global_linked_reports_df["report_file_exists"].any():
    print("No local report text files were found to preview.")
else:
    sample_report_row = global_linked_reports_df.query("report_file_exists").iloc[0]
    print(
        "Sample report: "
        f"subject_id={sample_report_row['subject_id']}, "
        f"study_id={sample_report_row['study_id']}"
    )
    print(read_report_text(mimic_cxr_dir, sample_report_row["report_path"]))


## Notes For The Next Modeling Pass

Use these outputs to decide whether the modular dataloader should preserve study-level grouping, whether paired frontal/lateral samples are common enough to require special batching, and whether the local machine has the image/report files required for multimodal experiments.
